---
## <font color='CornflowerBlue'>Practical 6: Visualising Token Probabilities in LLM Outputs</font>
##### Marieke Smith, Alok Bharadwaj, and Arjen Jakobi
---

In the previous notebook, we studied model confidence in a constrained setting: multiple-choice questions, where the model had to choose between a fixed set of answer options. In this notebook, we move to **open-ended generation**.

Open-ended answers are harder to evaluate because there is not always a single correct output. Instead of asking only whether an answer is right or wrong, we will inspect the model's **token probabilities**: how confident the model is about each generated token. By visualising these probabilities, we can examine where the model is highly certain, where it is uncertain, and how that uncertainty changes across an answer.

Run the first cell to initialise the environment and load the required packages.

In [ ]:
# #@title Click here to initialize the environment (Required) { display-mode: "form" }
import os, sys, subprocess

print("Initializing environment. Please wait.")

# Clone the git repo 
subprocess.run(["git", "clone", "https://github.com/cryoTUD/ai4nanobiology.git"], stdout=subprocess.DEVNULL)
os.chdir("ai4nanobiology/week_6")
sys.path.append(os.path.abspath('.'))

# Installing requirements
subprocess.run(["pip", "install", "-q", "sentencepiece", "protobuf", "tiktoken", "transformers", "accelerate"])
print("Environment ready! You can now start the exercise.")

In [ ]:
import requests
import numpy as np
import pandas as pd
import re
import math
from IPython.display import HTML, display

from src.utils import (
    setup_client,
    query_model,
    colored,
    hex_to_rgb,
    prob_to_color,
)

### Setting Up the Hugging Face Client

The helper functions used in this notebook send prompts to an open-weight Llama model through the Hugging Face inference infrastructure. Run the next cell to set up the client that will be used for all model queries.

In [ ]:
hf_client = setup_client()

## Exercise 1: Creating open-ended test questions

In the previous notebook, we used multiple-choice questions because they provide a fixed set of answer options. Here, we will work with **open-ended questions**, where the model must generate a free-form answer.

Create two small test sets, each containing five questions ordered from easy to difficult:

1. A test set that you design yourself. Think carefully about which questions should be easy or difficult for a language model.
2. A second test set generated by an LLM of your choice, also ordered from easy to difficult.

These two sets will allow us to compare your intuition about question difficulty with the model's own behaviour. Save the questions as lists in the cell below.

In [ ]:
# TO DO
self_made_test_set = []
# TO DO
generated_test_set = []

## Exercise 2: Visualising confidence in generated answers

In this exercise, we will colour-code the answers generated by a language model using the probabilities assigned to its output tokens. A token with a high probability was relatively easy for the model to predict in that context, while a token with a lower probability indicates more uncertainty.

We will use predefined helper functions to convert token probabilities into colours. Before completing the main functions, you will first experiment with how probabilities are mapped to visual output.

### 2a. Mapping a probability to a colour

Start with a simple example. Choose a confidence value between 0 and 1 and observe how the sentence colour changes. This will help you interpret the colour scale used later in the notebook.

In [ ]:
sentence = "The color of this sentence shows how confident I am that I can finish this exercise correctly"

# TO DO rate your confidence in the sentence as a number between 0 and 1
confidence = 
bg_color_hex = prob_to_color(confidence)
colored_sentence = colored(sentence, bg_color_hex)
display(HTML(colored_sentence))

### 2b. Inspecting the model response
Before we can colour-code generated text, we need to understand the structure of the response returned by the model. The next cell sends a prompt to the Llama-8B model and stores several parts of the response, including the generated answer, token probabilities, and log probabilities.

Run the cell and inspect the printed outputs. Pay particular attention to which data structure contains both the generated tokens and their associated probabilities, because that structure will be used in the functions below.


In [ ]:
prompt = 'If AI could dream, what would they dream about? Answer in a max of 5 sentences.'
model_name = "llama-8b"
max_tokens = 200
client = hf_client

response = query_model(prompt, model_name, max_tokens, client)
answer = response['answer']
logprobs = response['logprobs'] 
token_probs = response['token_probs']
logprobs_content = response['logprobs_content']

print(f"ANSWER: \n{answer}\n")
print('-'*100)
print(f"LOGPROBS: \n{logprobs}\n")
print('-'*100)
print(f"TOKEN_PROBS: \n{token_probs}\n")
print('-'*100)
print(f"LOGPROBS_CONTENT: \n{logprobs_content}\n")

### 2c. __Task__: Colour-coding by sentence

We will first colour-code the answer at the level of complete sentences. For each sentence, the function will combine the probabilities of the tokens in that sentence and compute an average token probability. The sentence is then displayed with a colour corresponding to that average.

Scan the function below and decide which response data structure contains the information needed for this task. Then complete the missing parts of the function.

In [ ]:
token_data =                                      # TO DO select the best datastructure

In [ ]:
def color_code_per_sentence(token_data):
    """
    Color codes the sentences of an LLM answer based on average token probability.
    
    Parameters
    ----------
    token_data : the input datastructure that contains the tokens and their token probabilities
    """
    colored_output = [] # to store the final output

    # we will build back each sentence by iterating over the input data
    # we also calculate the avg_prob per sentence
    # based on these avg_probs, we will give the sentence a background color
    current_sentence = ""
    current_sentence_probs = [] # to store the avg_prob values
    
    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob = math.exp(logprob)
        
        # Keep adding tokens to the current sentence
        # Also keep track of their probabilities
        current_sentence += token
        current_sentence_probs.append(prob)
        
        # We stop adding tokens to the sentence if the sentence ends
        # So check if the token contains a sentence-ending punctuation mark
        if any(punct in token for punct in ['.', '?', '!']):
            
            # Calculate average probability for this specific sentence
            avg_prob =                                                  # TO DO
            
            # Map the average probability to a hex color
            bg_color_hex =                                              # TO DO
            prob_percent = f"{avg_prob * 100:.2f}%"
            
            # Clean up leading spaces for printing and apply color
            clean_sentence =                                            # TO DO
            colored_sentence = f"{...} {prob_percent}"                  # TO DO 
            
            # Save to output
            colored_output.append(...)                                  # TO DO
            
            # Reset lists for the next sentence
            current_sentence =                                          # TO DO 
            current_sentence_probs =                                    # TO DO

    # Catch-all for any leftover text at the end that didn't end in punctuation
    if current_sentence.strip():
        avg_prob =                                                      # TO DO
        bg_color_hex =                                                  # TO DO                                                
        prob_percent = f"{avg_prob * 100:.2f}%"
        
        clean_sentence =                                                # TO DO
        colored_sentence = f"{} {prob_percent}"                         # TO DO

        # Save to output
                                                                        # TO DO

    print("--- Colored Confidence Output ---")
    display(HTML("<br>".join(colored_output)))
    print("-" * 50)

#### Testing sentence-level colour coding

Test your completed `color_code_per_sentence()` function by passing in the data structure you selected above. The output should display each sentence with a colour that reflects its average token probability.

In [ ]:
token_data =                                                # TO DO
color_code_per_sentence(token_data)

### 2d. __Task__: Colour coding by token

Sentence-level colouring provides a compact summary, but it can hide local variation within a sentence. We will now colour-code the answer **token by token**, so that each generated token is displayed according to its own probability.

Complete the function below by converting each token's log probability into a probability and mapping that probability to a colour.

In [ ]:
def color_code_per_token(token_data):
    """
    Color codes an LLM answer based on individual token probabilities.
    
    Parameters
    ----------
    token_data : containing the tokens and their logprobs
    """
    colored_tokens = [] # to store the individually colored tokens

    for item in token_data:
        token = item['token']
        logprob = item['logprob']
        
        # Skip hidden stop tokens like <|eot_id|>
        if token == "<|eot_id|>":
            continue
            
        # Convert logprob to normal probability
        prob =                                                            # TO DO
        
        # Map the probability to a hex color
        bg_color_hex =                                                    # TO DO
        
        # Apply the color directly to the token
        colored_token =                                                   # TO DO
        
        # Save to output
        colored_tokens.append(...)                                        # TO DO
    print("--- Colored Confidence Output (Per Token) ---")
    
    display(HTML("".join(colored_tokens)))                                # TO DO
    
    print("-" * 50)

#### Testing token-level colour-coding

Run the next cell to test `color_code_per_token()`. Compared with sentence-level colouring, token-level colouring should make it easier to see exactly where the model became more or less uncertain during generation.


In [ ]:
color_code_per_token(token_data)

### 2e. Comparing Confidence Across Your Test Questions

Now apply your colour coding functions to the open-ended question sets you created earlier. For each generated answer, compare the sentence-level and token-level visualisations.

As you inspect the results, consider the following questions:

1. Do the questions you expected to be difficult lead to lower token probabilities?
2. Does the LLM-generated test set contain questions that are clearly easier or harder than your own?
3. Are low-probability tokens associated with factual uncertainty, unusual phrasing, rare words, or something else?
4. Does high token probability always mean that the answer is factually correct?

Remember that token probability measures how predictable a token is under the model, not whether the generated statement is true. A model can be very confident about fluent but incorrect text, so these visualisations should be interpreted as a window into model uncertainty rather than as a direct measure of factual accuracy.

Write your observations in the cell below before running the remaining comparison cells.

In [ ]:
# My answer:

In [ ]:
model_name = "llama-8b"
max_tokens = 200
client = hf_client

for question in generated_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_sentence(token_data)

In [ ]:
for question in generated_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_token(token_data)

In [ ]:
for question in self_made_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_sentence(token_data)

In [ ]:
for question in self_made_test_set:
    prompt = question + ' Answer in a maximum of 5 sentences.'
    response = query_model(prompt, model_name, max_tokens, client)
    token_data = response['logprobs_content']
    color_code_per_token(token_data)